# Ex4 · TinyViT

In [ ]:
from __future__ import annotations

import math

import os

from dataclasses import dataclass, replace

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms

In [ ]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    if x.ndim != 4 or x.shape[-2] % patch_size or x.shape[-1] % patch_size:
        raise ValueError(
            "images must have shape (B,C,H,W) with H and W divisible by patch_size"
        )
    patches = x.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    return patches.permute(0, 2, 3, 1, 4, 5).reshape(
        x.shape[0], -1, x.shape[1] * patch_size**2
    )

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.projection = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.projection(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.embedding = nn.Parameter(torch.empty(1, num_tokens, d_model))
        nn.init.trunc_normal_(self.embedding, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] != self.embedding.shape[1]:
            raise ValueError("token count does not match positional embedding")
        return x + self.embedding

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        if variant not in {"geglu", "swiglu"}:
            raise ValueError(f"unknown GLU variant: {variant}")
        self.variant = variant
        self.input_projection = nn.Linear(d_model, 2 * d_ff_gated)
        self.output_projection = nn.Linear(d_ff_gated, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, value = self.input_projection(x).chunk(2, dim=-1)
        activated = F.gelu(gate) if self.variant == "geglu" else F.silu(gate)
        return self.dropout(self.output_projection(self.dropout(activated * value)))

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.norm1, self.norm2 = (nn.LayerNorm(d_model), nn.LayerNorm(d_model))
        self.attention = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.mlp = mlp
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm1(x)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        x = x + self.dropout(attended)
        return x + self.mlp(self.norm2(x))

In [ ]:
class TinyViT(nn.Module):
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        if 28 % patch_size:
            raise ValueError("patch_size must divide 28")
        self.num_tokens = (28 // patch_size) ** 2
        self.patch_size = patch_size
        self.patch_embed = PatchEmbed(patch_size**2, d_model)
        self.position = PositionalEmbedding(self.num_tokens, d_model)
        blocks = []
        for _ in range(n_layers):
            if mlp_kind == "ffn":
                mlp = FeedForward(d_model, d_ff, dropout)
            elif mlp_kind in {"geglu", "swiglu"}:
                mlp = GLUFeedForward(
                    d_model, max(1, int(2 * d_ff / 3)), dropout, mlp_kind
                )
            else:
                raise ValueError(f"unknown mlp_kind: {mlp_kind}")
            blocks.append(TransformerEncoderBlock(d_model, n_heads, mlp, dropout))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.position(self.patch_embed(patchify(x, self.patch_size)))
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x).mean(dim=1))

In [ ]:
images = torch.rand(2, 1, 28, 28)
for block in ("ffn", "geglu", "swiglu"):
    network = TinyViT(4, 32, 4, 1, 128, 0.1, block)
    print(block, network(images).shape)

In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 0.0003
    weight_decay: float = 0.01
    device: str = "cpu"


def train_one_run(mlp_kind, model, train_loader, test_loader, cfg):
    torch.manual_seed(cfg.seed)
    model.to(cfg.device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    losses, scores = [], []
    for epoch in range(cfg.epochs):
        model.train()
        for images, targets in train_loader:
            logits = model(images.to(cfg.device))
            loss = F.cross_entropy(logits, targets.to(cfg.device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        hits = examples = 0
        with torch.inference_mode():
            for images, targets in test_loader:
                prediction = model(images.to(cfg.device)).argmax(1)
                hits += int((prediction == targets.to(cfg.device)).sum())
                examples += len(targets)
        scores.append(hits / examples)
        print(f"{mlp_kind} seed={cfg.seed} epoch={epoch + 1} accuracy={scores[-1]:.4f}")
    return dict(
        variant=mlp_kind,
        seed=cfg.seed,
        train_losses=losses,
        test_accuracies=scores,
        parameters=sum(p.numel() for p in model.parameters()),
        best_test_accuracy=max(scores),
        final_test_accuracy=scores[-1],
    )

세 구조를 MNIST에서 학습하고 비교하는 코드는 [train_variants.py](../../experiments/hw1/train_variants.py)에 있습니다.

In [ ]:
from statistics import mean, stdev
import matplotlib.pyplot as plt

smoke = os.environ.get("HW_SMOKE") == "1"
device = os.environ.get("HW_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")


def loader(training):
    data = datasets.MNIST(
        "data", train=training, transform=transforms.ToTensor(), download=True
    )
    if smoke:
        data = Subset(data, range(512 if training else 256))
    return DataLoader(data, batch_size=128, shuffle=training)


train_loader, test_loader = loader(True), loader(False)
dimensions = (4, 32, 4, 1, 128, 0.1) if smoke else (4, 64, 4, 2, 256, 0.1)
seeds = range(2 if smoke else 3)
results = []
for seed in seeds:
    for kind in ("ffn", "geglu", "swiglu"):
        torch.manual_seed(seed)
        net = TinyViT(*dimensions, kind)
        cfg = TrainConfig(seed=seed, epochs=1 if smoke else 5, device=device)
        results.append(train_one_run(kind, net, train_loader, test_loader, cfg))
for kind in ("ffn", "geglu", "swiglu"):
    runs = [r for r in results if r["variant"] == kind]
    final = [r["final_test_accuracy"] for r in runs]
    print(
        f"{kind}: {runs[0]['parameters']} parameters, accuracy {mean(final):.2%} ± {stdev(final):.2%}"
    )
    curve = [mean(scores) for scores in zip(*(r["test_accuracies"] for r in runs))]
    plt.plot(range(1, len(curve) + 1), curve, label=kind)
plt.xlabel("Epoch")
plt.ylabel("Mean test accuracy")
plt.legend()
plt.show()